# GEOL 230 — Case Study 1a: Yellowstone National Park
## Student starter notebook (v3)

This notebook builds **Tables 1–3**, **Figures 1–4** and the **ArcGIS point file** from *your* cleaned Excel workbook.

**It will not run from top to bottom as posted. That is on purpose.** Each stop is a place where you show that you understand what the code is doing. There are four kinds of stop:

| You will see | What you do |
|---|---|
| **Fill in →** a blank written as `____` | Replace the blank with the right value, column name or method. If you leave one, Python stops with `NameError: name '____' is not defined`. |
| **Fix it →** a cell with a mistake built in (six of them) | Run it, read the error or look hard at the result, find the cause, repair it, and write one sentence saying what was wrong. Some of these do **not** raise an error — they quietly give a wrong answer, which is the more dangerous kind. |
| **Predict →** | Commit to a number *before* you run the cell, then compare. Being wrong is fine; not guessing is not. |
| **Explain →** | Answer in one or two of your own sentences, inside the quotation marks. These are graded for the idea, not the grammar. |

Cells that begin with `# CHECK` test your work. **Do not edit or delete them.** If a check prints `NOT YET`, read its message — it tells you what to look at, not what to type.

**Rules of the road**
1. Run the cells in order. Do not skip a cell that is failing.
2. When something breaks, read the **last line** of the red text first (Starter Notebook, Section 10).
3. You may use AI tools to help you debug. Paste the error and the cell, ask what it means, and record it in the AI Use Log at the bottom. You are responsible for every line you submit.
4. Before you submit: **Runtime → Restart session and run all**. It must finish with no errors.

**What this notebook saves into `outputs/`**

| File | What it is |
|---|---|
| `Table1_YNP_SoilChem_2016.xlsx` | 2016 soil chemistry: mean and SD for pH, EC, organic matter, bulk density, total carbon |
| `Figure1_SoilChem_BulkParams.png` | pH, EC, bulk density — bars clustered by parameter, SD error bars |
| `Figure2_SoilChem_Carbon.png` | organic matter and total carbon — clustered by parameter, SD error bars |
| `Table2_YNP_SoilTemp_YearlyAvg.xlsx` | yearly average soil temperature for each summit |
| `Figure3_SoilTemp_E22_Daily.png` | daily temperature, plot E22, all four summits, one year |
| `Table3_BechlerRiver_WaterChem_Stats.xlsx` | Bechler River: min, max, mean, SD and sample counts |
| `Figure4_WaterChem_NewFigureType.png` | a water-chemistry figure of a new type, with a caption |
| `Map1_points_for_arcgis.csv` | point file for ArcGIS Pro |

## 0 · Setup

Run this cell as it is. It loads the packages and defines a few helpers (`save_df_excel`, `save_fig`, `checkpoint`, `check`, `check_answer`). You do not need to change anything here except your name.

In [ ]:
STUDENT_NAME = "____"        # Fill in -> your name, e.g. "Alex Smith"

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

plt.rcParams["figure.dpi"] = 110

OUTPUT_DIR = "outputs"                       # a folder next to this notebook
os.makedirs(OUTPUT_DIR, exist_ok=True)


def save_df_excel(df, filename, sheet_name="Table", *, index=True):
    # Save a DataFrame as an Excel file in outputs/
    path = os.path.join(OUTPUT_DIR, filename)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index)
    print(f"Saved: {path}")


def save_fig(filename, fig=None):
    # Save a figure (the current one by default) in outputs/ at 300 dpi.
    # Refuses if a label still has a blank in it.
    fig = fig if fig is not None else plt.gcf()
    texts = []
    for ax in fig.axes:
        texts += [ax.get_xlabel(), ax.get_ylabel(), ax.get_title()]
        leg = ax.get_legend()
        if leg is not None:
            texts.append(leg.get_title().get_text())
    if any("____" in t for t in texts):
        raise AssertionError("NOT YET - this figure still has a '____' in a label, title or legend. "
                             "A figure with no labels or units is not finished.")
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Saved: {path}")


def checkpoint(title, df, cols=None, n=5):
    # Print the shape and a few values so you can see what a step did
    print(f"\n--- {title} ---")
    print("shape (rows, columns):", df.shape)
    for c in (cols or []):
        if c in df.columns:
            print(f"  {c} unique values (first 10):", list(pd.Series(df[c].dropna().unique())[:10]))
    display(df.head(n))


def check(condition, message):
    # Used by the CHECK cells
    if not condition:
        raise AssertionError("NOT YET - " + message)
    print("CHECK PASSED")


def check_answer(label, text, min_words=8):
    # Used for the Explain and Fix-it sentences
    words = str(text).replace("____", " ").split()
    if len(words) < min_words:
        raise AssertionError(f"NOT YET - {label}: write your answer inside the quotation marks "
                             f"(at least {min_words} words, in your own words).")
    print(f"{label}: recorded ({len(words)} words)")


check(STUDENT_NAME.strip() not in ("", "____"), "put your name in the cell above.")
print("Notebook of:", STUDENT_NAME)

## 1 · Load your cleaned workbook

Upload **your own** workbook (`LASTNAME_FIRSTNAME_GEOL230_YNP_CaseStudy.xlsx`). In Colab: click the folder icon on the left and drag the file in, or let the upload button below appear and choose the file.

In [ ]:
FILE_NAME = "LASTNAME_FIRSTNAME_GEOL230_YNP_CaseStudy.xlsx"    # change to your file name

if not os.path.exists(FILE_NAME):
    from google.colab import files           # shows an upload button in Colab
    uploaded = files.upload()
    FILE_NAME = list(uploaded.keys())[0]

xls = pd.ExcelFile(FILE_NAME)
print("Using:", FILE_NAME)
print("Tabs in this workbook:")
for s in xls.sheet_names:
    print("  -", s)

**Fill in →** the three *clean* tab names, exactly as printed above (spaces and parentheses count).

In [ ]:
SOIL_SHEET  = "____"
TEMP_SHEET  = "____"
WATER_SHEET = "____"

soil_df  = pd.read_excel(FILE_NAME, sheet_name=SOIL_SHEET)
temp_df  = pd.read_excel(FILE_NAME, sheet_name=TEMP_SHEET)      # the big one: give it a few seconds
water_df = pd.read_excel(FILE_NAME, sheet_name=WATER_SHEET)

checkpoint("soil_df", soil_df, cols=["SiteName", "YearSampled"])
checkpoint("temp_df", temp_df, cols=["Park", "Summit", "Plot"])
checkpoint("water_df", water_df, cols=["Area", "Type"])

The next cell checks the Excel cleaning you did on Day 1. If it says `NOT YET`, the fix is in **Excel**, not here: repair the Clean tab, save, upload the workbook again, and re-run from the top of Section 1.

In [ ]:
# CHECK 1 - is the workbook really clean?  (do not edit)
check(soil_df["SiteName"].astype(str).str.contains("YNP").all(),
      "Clean Soil Chem still contains sites outside Yellowstone. Filter it in Excel.")
check((temp_df["Park"] == "YELL").all(),
      "Clean Soil Temp still contains other parks. Filter it in Excel.")

_water_text = water_df.astype(str)
_n_lt = int(_water_text.apply(lambda c: c.str.contains("<", regex=False)).values.sum())
_n_dash = int(_water_text.apply(lambda c: c.str.fullmatch(r"\s*-+\s*")).values.sum())
check(_n_lt == 0, f"Clean Water Chem still has {_n_lt} cells containing '<'. Flag them, then remove the symbol in Excel.")
check(_n_dash == 0, f"Clean Water Chem still has {_n_dash} cells that are only dashes. Make them blank (not 0) in Excel.")
print("Workbook looks clean. Rows: soil", len(soil_df), "| temperature", len(temp_df), "| water", len(water_df))

## 2 · What shape are the soil data?

Look at the `soil_df` preview above before you go on.

**Fix it → (1 of 6)** The cell below tries to pull out the soil pH values. Run it. Read the last line of the error. Then replace the line so that `ph_rows` holds **every row of `soil_df` that is a pH measurement**.

In [ ]:
# FIX IT 1
ph_rows = soil_df["pH"]

display(ph_rows)

In [ ]:
# CHECK 2  (do not edit)
check(isinstance(ph_rows, pd.DataFrame) and len(ph_rows) > 0 and (ph_rows["ParameterDataset"] == "pH").all(),
      "ph_rows should be a table of rows, all of them pH. Which column holds the word 'pH'?")

# Fix-it note: what was wrong, and how did the error message tell you?
fix_1 = "____"
check_answer("Fix it 1", fix_1)

# Explain -> This layout is called 'long format'. In your own words: what is one ROW of soil_df?
#            (What would the same data look like in 'wide format'?)
explain_1 = "____"
check_answer("Explain 1", explain_1, min_words=12)

## 3 · The four pandas moves, on a toy dataset first

Everything in this case study is built from four moves:

1. **Filter rows** with a condition: `df[df["col"] == value]`
2. **Keep certain categories** with `.isin([...])`
3. **Translate values** with `.map({...})`
4. **Turn text into numbers** with `pd.to_numeric(..., errors="coerce")`

The demo cell runs as it is. Read it like a recipe and, after each step, ask: *did the rows change? did the columns change?*

In [ ]:
# DEMO - runs as is
toy = pd.DataFrame({
    "YearSampled": [2016, 2016, 2015, 2016, 2017, 2016, 2016, 2016],
    "SiteName": ["YNP-01", "YNP-02", "YNP-01", "ABC-99", "YNP-03", "YNP-04", None, "YNP-05"],
    "ParameterDataset": ["pH", "EC", "pH", "pH", "C", "Bulk_Density", "OrganicMatter", "EC"],
    "Average": ["7.1", "120", "6.8", "8.0", "1.9", "0.92", "n/a", "115"],
    "StDev": ["0.2", "5", "0.1", "0.3", "0.4", "0.05", "0.8", "bad"],
})
TOY_MAP = {"pH": "pH", "EC": "EC", "OrganicMatter": "Organic Matter",
           "Bulk_Density": "Bulk Density", "C": "Total Carbon"}

checkpoint("Toy data, start", toy, n=8)

demo = toy[toy["YearSampled"] == 2016]                                   # move 1: filter by year
checkpoint("after year filter", demo, cols=["YearSampled"], n=8)

demo = demo[demo["SiteName"].astype(str).str.contains("YNP")]            # move 1 again: filter by text
checkpoint("after site filter", demo, cols=["SiteName"], n=8)

demo = demo[demo["ParameterDataset"].isin(TOY_MAP.keys())].copy()        # move 2: keep listed categories
demo["Parameter"] = demo["ParameterDataset"].map(TOY_MAP)                # move 3: translate
demo["Mean"] = pd.to_numeric(demo["Average"], errors="coerce")           # move 4: text -> number
demo["SD"]   = pd.to_numeric(demo["StDev"], errors="coerce")
checkpoint("after isin + map + to_numeric", demo[["SiteName", "Parameter", "Mean", "SD"]], n=8)

**Predict →** In the demo, one value in the `SD` column became `NaN`. Which original value was it, and why did `errors="coerce"` do that instead of stopping with an error? (Answer in the next cell.)

**Fill in →** Now a slightly different question on the same toy data: keep **2016**, keep sites whose name contains **"YNP"**, and keep **only pH and EC**. Before you run it, predict how many rows will be left.

In [ ]:
predict_toy_rows = ____          # Predict -> a whole number, BEFORE you run this cell

toy2 = toy.copy()
toy2 = toy2[toy2["YearSampled"] == ____]
toy2 = toy2[toy2["SiteName"].astype(str).str.contains("____")]
toy2 = toy2[toy2["ParameterDataset"].isin([____, ____])].copy()
toy2["Mean"] = pd.to_numeric(toy2["____"], errors="____")
toy2["SD"]   = pd.to_numeric(toy2["____"], errors="____")

display(toy2)
print("You predicted", predict_toy_rows, "rows; there are", len(toy2))

# Explain -> which SD value became NaN, and why is NaN better here than an error or a zero?
explain_2 = "____"

In [ ]:
# CHECK 3  (do not edit)
check(len(toy2) == 3 and set(toy2["ParameterDataset"]) == {"pH", "EC"},
      "toy2 should have 3 rows (2016, YNP sites, pH or EC only). Run checkpoint(...) after each filter to see which step is off.")
check(toy2["Mean"].dtype.kind == "f" and toy2["SD"].dtype.kind == "f" and toy2["SD"].isna().sum() == 1,
      "Mean and SD should be numbers, with exactly one NaN in SD.")
check_answer("Explain 2", explain_2, min_words=12)

## 4 · Table 1 and Figures 1–2 — soil chemistry, 2016

**Requirements**
- **Table 1:** one row per summit; mean and standard deviation of **pH, EC, organic matter, bulk density, total carbon**; **2016 only**.
- **Figure 1:** clustered bar chart for pH, EC and bulk density; bars clustered by **parameter**; SD error bars.
- **Figure 2:** the same for organic matter and total carbon.

The Clean Soil Chem tab already has an `Average` and a `StDev` for each summit, campaign and parameter. You will **report** those; you will not compute a new standard deviation.

**Fix it → (2 of 6)** The cell below should leave exactly the rows Table 1 needs. It has one blank and one mistake that does **not** raise an error. Fill in the blank, run it, and use CHECK 4 to find out whether you have the right rows.

In [ ]:
# FIX IT 2
PARAM_MAP = {
    "pH": "pH",
    "EC": "EC",
    "Organic Matter": "Organic Matter",
    "Bulk_Density": "Bulk Density",
    "TC": "Total Carbon",
}

soil2016 = soil_df.copy()
soil2016 = soil2016[soil2016["YearSampled"] == ____]
checkpoint("after year filter", soil2016, cols=["YearSampled"], n=3)

soil2016 = soil2016[soil2016["ParameterDataset"].isin(PARAM_MAP.keys())].copy()
checkpoint("after parameter filter", soil2016, cols=["ParameterDataset"], n=3)

In [ ]:
# CHECK 4  (do not edit)
_n = len(soil2016)
if _n == 0:
    _hint = "no rows survived. Print soil_df['YearSampled'].unique() - is the year a number or text in your data?"
elif soil2016["YearSampled"].nunique() > 1:
    _hint = (f"you have {_n} rows from more than one year. If these went into Table 1, pandas would average the "
             "years together without telling you.")
else:
    _hint = (f"you have {_n} rows. Table 1 needs 4 summits x 5 parameters. Count how many DIFFERENT parameters "
             "survived the filter, then print soil_df['ParameterDataset'].unique() and compare it, character "
             "by character, with the keys of PARAM_MAP.")
check(_n == 20 and soil2016["ParameterDataset"].nunique() == 5 and (soil2016["YearSampled"] == 2016).all(), _hint)

# Fix-it note: what was wrong, and why did Python not complain about it?
fix_2 = "____"
check_answer("Fix it 2", fix_2)

**Fill in →** Add three columns: a readable `Parameter` name (move 3), and numeric `Mean` and `SD` (move 4).

In [ ]:
soil2016["Parameter"] = soil2016["ParameterDataset"].____(PARAM_MAP)
soil2016["Mean"] = pd.to_numeric(soil2016["____"], errors="coerce")
soil2016["SD"]   = pd.to_numeric(soil2016["____"], errors="coerce")

display(soil2016[["SiteName", "Parameter", "UnitDataset", "Count", "Mean", "SD"]])

### Table 1: long → wide

`pivot_table` turns the 20 long rows into a 4 × 5 table. The means are done for you. **Fill in →** write the line for the standard deviations by analogy.

In [ ]:
COL_ORDER = ["pH", "EC", "Organic Matter", "Bulk Density", "Total Carbon"]

t1_mean = soil2016.pivot_table(index="SiteName", columns="Parameter", values="Mean").reindex(columns=COL_ORDER)
t1_sd   = ____

table1 = pd.concat({"Mean": t1_mean, "SD": t1_sd}, axis=1)
display(table1)
save_df_excel(table1, "Table1_YNP_SoilChem_2016.xlsx", sheet_name="Table1")

In [ ]:
# CHECK 5  (do not edit)
check(table1.shape == (4, 10) and not table1.isna().any().any(),
      "Table 1 should be 4 summits x 10 columns (5 means + 5 SDs) with no empty cells. "
      "If a whole column is NaN, its name in COL_ORDER does not match the names made by PARAM_MAP.")
check(abs(table1.loc["US_YNP_BLP", ("Mean", "pH")] - 6.57) < 0.005 and abs(table1.loc["US_YNP_BLP", ("SD", "pH")] - 0.60) < 0.005,
      "Compare your BLP pH with the Table 1 slide from class (6.57 +/- 0.60). A different mean usually means "
      "more than one year is being averaged; a wrong SD means t1_sd is not built from the SD column.")

# Explain -> Table 1 reports the StDev column from the file. A classmate instead computes
#            soil2016.groupby("Parameter")["Mean"].std(). What does THEIR number measure,
#            and why is it not the SD that belongs in Table 1?
explain_3 = "____"
check_answer("Explain 3", explain_3, min_words=15)

### Figure 1 — pH, EC and bulk density

"Clustered by parameter" means the **parameters** are on the x-axis and each summit is a colored bar inside each cluster. The whiskers are the standard deviations from Table 1.

**Fix it → (3 of 6)** Run the cell and **look at the figure** before you read anything else. Something a reader needs is missing or wrong, and Python may not tell you. Find the cause, repair it, and fill in the label blanks.

In [ ]:
# FIX IT 3
FIG1_PARAMS = ["pH", "EC", "Bulk Density"]
f1 = soil2016[soil2016["Parameter"].isin(FIG1_PARAMS)]

mean_w = f1.pivot(index="Parameter", columns="SiteName", values="Mean").reindex(FIG1_PARAMS)
sd_w   = f1.pivot(index="SiteName", columns="Parameter", values="SD")

ax = mean_w.plot(kind="bar", yerr=sd_w, capsize=3, rot=0, figsize=(7, 4))
ax.set_xlabel("____")
ax.set_ylabel("____")
ax.legend(title="____")
plt.tight_layout()
plt.show()

In [ ]:
# CHECK 6  (do not edit)
check(sd_w.shape == mean_w.shape and list(sd_w.index) == list(mean_w.index) and list(sd_w.columns) == list(mean_w.columns),
      "the table of whiskers (sd_w) must have the same rows and columns, in the same order, as the table of bar "
      "heights (mean_w). Display both tables and compare them.")
check(ax.get_ylabel().strip() not in ("", "____") and ax.get_xlabel().strip() != "____",
      "label the axes. What do the bar heights represent?")

# Fix-it note: what was missing from the first version of the figure, and what caused it?
fix_3 = "____"
check_answer("Fix it 3", fix_3)

**Explain →** Look at your repaired figure. The EC bars are almost invisible. Is electrical conductivity really close to zero at these summits? Say what is going on, and what you would change about the figure so that it does not mislead.

Then decide: keep the single chart, or rebuild Figure 1 as **three panels, one parameter per panel** (preferred — `fig, axes = plt.subplots(1, 3)`, then one `ax.bar(..., yerr=...)` per panel, each with its own y-label and unit). Either earns full credit if it is labeled and captioned. Put your final version in the cell below; it is the one that gets saved.

In [ ]:
explain_4 = "____"
check_answer("Explain 4", explain_4, min_words=15)

# Your final Figure 1 goes here. To keep the single chart, re-draw it with these two lines.
# To build the three-panel version, replace them with your own code.
ax = mean_w.plot(kind="bar", yerr=sd_w, capsize=3, rot=0, figsize=(7, 4))
ax.set_xlabel("____"); ax.set_ylabel("____"); ax.legend(title="____")

plt.tight_layout()
save_fig("Figure1_SoilChem_BulkParams.png")
plt.show()

FIG1_CAPTION = "____"       # Figure 1. ... (what the bars are, what the whiskers are, year, n)
check_answer("Figure 1 caption", FIG1_CAPTION, min_words=12)

### Figure 2 — organic matter and total carbon

**Your turn.** No scaffold this time: Figure 2 is Figure 1's recipe with two parameters instead of three. Both are percentages, so one y-axis works. Write the code between the two comment lines. It must end with the figure saved as `Figure2_SoilChem_Carbon.png`.

In [ ]:
FIG2_PARAMS = ["____", "____"]

# ---- your code starts here: filter -> pivot the means -> pivot the SDs -> plot with yerr -> labels ----


# ---- your code ends here ----
check(sum(len(a.patches) for a in plt.gcf().axes) >= 8, "nothing is drawn yet: Figure 2 needs 8 bars (2 parameters x 4 summits).")
plt.tight_layout()
save_fig("Figure2_SoilChem_Carbon.png")
plt.show()

FIG2_CAPTION = "____"

In [ ]:
# CHECK 7  (do not edit)
check(set(FIG2_PARAMS) == {"Organic Matter", "Total Carbon"},
      "FIG2_PARAMS should hold the two readable parameter names created by PARAM_MAP.")
check_answer("Figure 2 caption", FIG2_CAPTION, min_words=12)

## 5 · Table 2 and Figure 3 — soil temperature

**Requirements**
- **Table 2:** yearly average soil temperature for each summit, from the daily data.
- **Figure 3:** daily temperature for **plot E22** at all four summits, for **one year or one 12-month window**, with a sentence saying why you chose it.

Each summit has four plots (E22, N22, S22, W22), and each plot has one `DailyMean` per day.

**Predict →** How many rows should **one summit** have in **one complete year**? Work it out before you run the cell.

**Fill in →** the blanks.

In [ ]:
predict_rows_summit_year = ____       # Predict -> rows for one summit in one full (non-leap) year

temp_df["DateTime"] = pd.to_datetime(temp_df["____"])
temp_df["Year"] = temp_df["DateTime"].dt.____

temp_yell = temp_df[temp_df["Park"] == "____"].copy()

table2 = (
    temp_yell.groupby(["____", "____"])["DailyMean"]
    .agg(YearlyMeanTemp_C="mean", n_days="count")
    .reset_index()
)
display(table2)
save_df_excel(table2, "Table2_YNP_SoilTemp_YearlyAvg.xlsx", sheet_name="Table2", index=False)

In [ ]:
# CHECK 8  (do not edit)
check(len(table2) == 40 and {"Summit", "Year"}.issubset(table2.columns),
      f"Table 2 should have one row per summit per year (4 summits x 10 years). You have {len(table2)} rows. "
      "What are you grouping by?")
_blp16 = table2[(table2["Summit"] == "BLP") & (table2["Year"] == 2016)].iloc[0]
check(abs(_blp16["YearlyMeanTemp_C"] - 1.20) < 0.01, "BLP 2016 should average 1.20 °C. Check the grouping and the park filter.")
print("BLP 2016:", round(_blp16["YearlyMeanTemp_C"], 2), "°C from", int(_blp16["n_days"]), "rows.",
      "You predicted", predict_rows_summit_year, "rows for a full year.")

# Explain -> Find the BLP 2011 row in Table 2. How many rows is its mean built from, and what does
#            that tell you about comparing the 2011 mean with the 2016 mean?
explain_5 = "____"
check_answer("Explain 5", explain_5, min_words=15)

### Figure 3 — daily temperature at plot E22

**Fix it → (4 of 6)** The cell below runs without an error and draws something. Run it and look. Then re-read the Figure 3 requirement, decide which rows the figure should actually contain, and change the line that builds `e22`. Fill in the label blanks too.

In [ ]:
# FIX IT 4
e22 = temp_yell.copy()

fig, ax = plt.subplots(figsize=(10, 4))
for summit, g in e22.groupby("Summit"):
    g = g.sort_values("DateTime")
    ax.plot(g["DateTime"], g["DailyMean"], label=summit, linewidth=1)

ax.set_xlabel("____")
ax.set_ylabel("____")
ax.legend(title="____")
plt.tight_layout()
plt.show()

In [ ]:
# CHECK 9  (do not edit)
check(list(e22["Plot"].unique()) == ["E22"],
      "Figure 3 is for one plot only. Print e22['Plot'].unique() to see what you are plotting.")
_span = (e22["DateTime"].max() - e22["DateTime"].min()).days
check(_span <= 366, f"your figure covers {_span} days. The requirement is one year or one 12-month window.")
check(e22["Summit"].nunique() == 4, "all four summits should appear.")
print("Days of data per summit in your window:", e22.groupby("Summit")["DailyMean"].count().to_dict())

# Fix-it note: what did the first version look like, and which two things did you have to filter?
fix_4 = "____"
check_answer("Fix it 4", fix_4)

# Explain -> why THIS year or window? (Look at the days-per-summit line printed above.)
FIG3_JUSTIFICATION = "____"
check_answer("Figure 3 justification", FIG3_JUSTIFICATION, min_words=12)

save_fig("Figure3_SoilTemp_E22_Daily.png", fig)
FIG3_CAPTION = "____"
check_answer("Figure 3 caption", FIG3_CAPTION, min_words=12)

## 6 · Table 3 and Figure 4 — Bechler River water chemistry

**Requirements**
- **Table 3:** for the **Bechler River** area: minimum, maximum, mean and standard deviation of the main water-chemistry variables, plus the **number of samples** behind each one (including discharge).
- **Figure 4:** a descriptive water-chemistry figure of a **new type** — not a clustered bar chart and not a time series like Figure 3 — with a short caption saying why that type fits.

The water tab has one extra row at the top that holds the **units**, not a sample.

**Fix it → (5 of 6)** Fill in the blank that removes the units row. Then run the cell: it does not raise an error, but look at what `bech` contains.

In [ ]:
# FIX IT 5
w = water_df.copy()
w = w[w["ID"].____()].copy()             # keep only rows that HAVE a sample ID (drops the units row)
w["Collection date"] = pd.to_datetime(w["Collection date"], errors="coerce")

bech = w[w["Area"] == "Bechler river"].copy()
checkpoint("Bechler River samples", bech[["ID", "Sample site", "Area", "Type", "Collection date"]])

In [ ]:
# CHECK 10  (do not edit)
check(len(w) == 66, f"after dropping the units row there should be 66 samples; you have {len(w)}.")
check(len(bech) == 28,
      f"bech has {len(bech)} rows; the Bechler River area has 28 samples. An empty table with no error almost "
      "always means the text you asked for does not exactly match the text in the data. Print w['Area'].unique().")

# Fix-it note: what was wrong, and why is an empty table more dangerous than an error message?
fix_5 = "____"
check_answer("Fix it 5", fix_5)

**Fill in →** Convert the variables to numbers, then build Table 3 with the four statistics the requirement names plus a count.

In [ ]:
CORE_VARS = [
    "Discharge", "pH", "Specific conductance", "Temperature",
    "Calcium (Ca)", "Magnesium (Mg)", "Sodium (Na)", "Potassium (K)",
    "Chloride (Cl)", "Sulfate (SO4)", "Alkalinity", "Silica (SiO2)",
    "Fluoride (F)", "Bromide (Br)", "Nitrate (NO3)", "Ammonium (NH4)",
    "Dissolved Organic Carbon (DOC)", "Phosphorus (P)", "Arsenic (As)",
]      # you may shorten this list for your Word table, but keep Discharge

for c in CORE_VARS:
    bech[c] = pd.to_numeric(bech[c], errors="____")

table3 = bech[CORE_VARS].agg(["____", "____", "____", "____", "count"]).T
table3 = table3.rename(columns={"count": "n_samples"})
display(table3)
save_df_excel(table3, "Table3_BechlerRiver_WaterChem_Stats.xlsx", sheet_name="Table3")

In [ ]:
# CHECK 11  (do not edit)
check({"min", "max", "mean", "std", "n_samples"}.issubset(table3.columns),
      "Table 3 needs min, max, mean, std and the count. Use the pandas names for those statistics.")
check(int(table3.loc["pH", "n_samples"]) == 28 and int(table3.loc["Discharge", "n_samples"]) == 13,
      "pH should have 28 samples and Discharge 13. If Discharge has 28, were the '---' cells turned into 0 in Excel?")
print("Temperature:", table3.loc["Temperature", "min"], "to", table3.loc["Temperature", "max"], "°C")

# Explain -> (a) Discharge has fewer samples than pH. Look at the Type column of bech and say why.
#            (b) The maximum temperature is about 90 °C in a table called 'Bechler River'.
#                What does that tell you about what the MEAN temperature in this table describes?
explain_6 = "____"
check_answer("Explain 6", explain_6, min_words=20)

### Figure 4 — your choice of a new figure type

**Your turn.** Choose a figure type that answers a question about the water chemistry. Some options:

| Type | Good for | Start with |
|---|---|---|
| Scatter plot | how two variables relate (try log axes; color or marker by `Type`) | `ax.scatter(x, y)` |
| Box plot or strip plot | comparing one variable between groups (`Type`, or `Area` using `w`) | `ax.boxplot([group1, group2])` |
| Histogram | the distribution of one variable | `ax.hist(values)` |
| Correlation heatmap | which variables rise and fall together | `ax.imshow(bech[vars].corr(), cmap="RdBu_r", vmin=-1, vmax=1)` |

Requirements: axis labels **with units** (the units row of your workbook has them), a legend or colorbar if color means something, and a caption that says what the figure shows **and why this type suits the question**. Write the code between the comment lines.

In [ ]:
FIG4_TYPE = "____"               # e.g. "scatter plot"

fig, ax = plt.subplots(figsize=(6.5, 5))
# ---- your code starts here ----


# ---- your code ends here ----
plt.tight_layout()
save_fig("Figure4_WaterChem_NewFigureType.png")
plt.show()

FIG4_CAPTION = "____"            # Figure 4. ... what it shows + why this figure type

In [ ]:
# CHECK 12  (do not edit)
_drawn = sum(len(a.lines) + len(a.collections) + len(a.patches) + len(a.images) for a in fig.axes)
check(_drawn > 0, "Figure 4 is empty. Draw something on ax before the figure is saved.")
check(FIG4_TYPE.strip() not in ("", "____") and "bar" not in FIG4_TYPE.lower(),
      "name your figure type. It cannot be a bar chart.")
check(any(a.get_xlabel().strip() or a.get_ylabel().strip() or a.images for a in fig.axes),
      "label your axes, with units.")
check_answer("Figure 4 caption", FIG4_CAPTION, min_words=20)

## 7 · The point file for ArcGIS Pro

This cell builds one row per mapped location: the four GLORIA summits (their coordinates come from the GLORIA site report, because the soil tabs have none) and one row per water sampling site. ArcGIS will read `Longitude` as X and `Latitude` as Y, in decimal degrees.

**Fix it → (6 of 6)** The cell runs and writes a file. Before you trust it, read the latitude and longitude ranges it prints and ask where on Earth those points would land.

In [ ]:
# FIX IT 6
GLORIA_SUMMIT_COORDS = {            # (latitude, longitude) in decimal degrees
    "US_YNP_BLP": (44.70028, 109.8267),
    "US_YNP_SCP": (44.69584, 109.8339),
    "US_YNP_SPP": (44.70167, 109.8350),
    "US_YNP_WSP": (44.69444, 109.8364),
}

gloria_points = pd.DataFrame({
    "site_id": list(GLORIA_SUMMIT_COORDS.keys()),
    "Latitude": [v[0] for v in GLORIA_SUMMIT_COORDS.values()],
    "Longitude": [v[1] for v in GLORIA_SUMMIT_COORDS.values()],
    "source": "GLORIA_soil",
    "Area": "GLORIA summits",
    "Type": "soil summit",
})

water_points = (
    w[["Sample site", "Latitude", "Longitude", "Area", "Type"]]
    .rename(columns={"Sample site": "site_id"})
    .assign(Latitude=lambda d: pd.to_numeric(d["Latitude"], errors="coerce"),
            Longitude=lambda d: pd.to_numeric(d["Longitude"], errors="coerce"),
            source="USGS_water")
    .dropna(subset=["Latitude", "Longitude"])
    .drop_duplicates(subset=["site_id"])
)

points = pd.concat([gloria_points, water_points], ignore_index=True)
print("Points:", len(points), "|", points["source"].value_counts().to_dict())
print("Latitude  from", points["Latitude"].min(), "to", points["Latitude"].max())
print("Longitude from", points["Longitude"].min(), "to", points["Longitude"].max())

In [ ]:
# CHECK 13  (do not edit)
check(points["Latitude"].between(43.5, 45.5).all() and points["Longitude"].between(-112.5, -109.0).all(),
      "at least one point is nowhere near Yellowstone (about 44-45° N, 110-111° W). "
      "Display points.sort_values('Longitude') and look at both ends. Which hemisphere is Wyoming in?")
check(len(points) == 63, f"expected 63 points (4 summits + 59 water sites); you have {len(points)}.")

points.to_csv(os.path.join(OUTPUT_DIR, "Map1_points_for_arcgis.csv"), index=False)
print("Saved:", os.path.join(OUTPUT_DIR, "Map1_points_for_arcgis.csv"))

# Fix-it note: what was wrong, and where would those four points have appeared on your map?
fix_6 = "____"
check_answer("Fix it 6", fix_6)

## 8 · Ethical AI Use Log (required)

Double-click this cell and fill in the table. One row per use. If you used no AI tools, write **No AI used** and one sentence on how you debugged instead.

| Tool and date | What I asked (short) | What I used or changed | How I checked it was right |
|---|---|---|---|
| | | | |
| | | | |

## 9 · Final check and download

Run this last. It lists what is in `outputs/`, prints every answer you wrote so you can proof-read them, and zips the folder for download. Then do **Runtime → Restart session and run all** one more time and submit.

Still to make outside this notebook: `Map1_YNP_StudyLocations.png` (ArcGIS Pro) and `Tables_All.docx` (Word).

In [ ]:
REQUIRED = [
    "Table1_YNP_SoilChem_2016.xlsx", "Table2_YNP_SoilTemp_YearlyAvg.xlsx",
    "Table3_BechlerRiver_WaterChem_Stats.xlsx", "Figure1_SoilChem_BulkParams.png",
    "Figure2_SoilChem_Carbon.png", "Figure3_SoilTemp_E22_Daily.png",
    "Figure4_WaterChem_NewFigureType.png", "Map1_points_for_arcgis.csv",
]
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(OUTPUT_DIR, f))]
for f in REQUIRED:
    print("OK      " if f not in missing else "MISSING ", f)

print("\n=== Your written answers,", STUDENT_NAME, "===")
ANSWERS = {
    "Fix it 1": fix_1, "Explain 1": explain_1, "Explain 2": explain_2, "Fix it 2": fix_2,
    "Explain 3": explain_3, "Fix it 3": fix_3, "Explain 4": explain_4, "Figure 1 caption": FIG1_CAPTION,
    "Figure 2 caption": FIG2_CAPTION, "Explain 5": explain_5, "Fix it 4": fix_4,
    "Figure 3 justification": FIG3_JUSTIFICATION, "Figure 3 caption": FIG3_CAPTION,
    "Fix it 5": fix_5, "Explain 6": explain_6, "Figure 4 type": FIG4_TYPE,
    "Figure 4 caption": FIG4_CAPTION, "Fix it 6": fix_6,
}
for k, v in ANSWERS.items():
    print(f"\n{k}:\n  {v}")

check(not missing, "some output files are missing: " + ", ".join(missing))

zip_name = "outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUTPUT_DIR)):
        z.write(os.path.join(OUTPUT_DIR, f), arcname=os.path.join("outputs", f))
print("\nZipped:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    pass